# Rerun 3 — SHAP explanations and faithfulness of the decision (Paket C; R1-6, R2-5)

This notebook re-runs the explainability experiment on the **seed-42 models of the rerun** (notebook 03: same engines, embedding cache, fold stratification and XGBoost settings) and answers the second-round comments:

| Comment | What is done here |
|---|---|
| R1-6: which target, what is "correct" | Prespecified target per text. SemEval: the emotion with the largest margin over its threshold, *c* = argmax (*P_c* − *τ_c*); the deployed decision is *P_c* ≥ *τ_c*. Tokopedia: the predicted class *k* = argmax (*α ⊙ P*); the explained function is the **decision-adjusted** score *q_k* = *α_k P_k* / Σ *α_j P_j*. "Correct" = target emotion is a gold label (SemEval) / target = gold class (Tokopedia). |
| R1-6: faithfulness of the final decision | Besides AOPC on the explained score, the **decision-flip rate** (deletion: target decision lost; insertion: target decision reached) is reported for SHAP vs. random rankings. Tokopedia also reports AOPC on the raw probability *P_k*. |
| R1-6: signed vs. absolute ranking | Every test is run with tokens ranked by \|φ\| and by signed φ (most positive first). |
| R1-6: masking, prefix, alignment | SHAP and the perturbation tests use the **same mask token `<mask>`** (the first version used "..." in SHAP and `<mask>` in AOPC). The prefix `query: ` is added after masking and is never masked. Tokens are regex word spans; a text is excluded only when the SHAP tokens and the spans disagree, and every exclusion is recorded. |
| R2-5: sample | Prespecified stratified sample of the seed-42 test sets: correctness (correct / error) × confidence tercile (low / mid / high), up to 8 texts per stratum and SemEval language and 15 per stratum on Tokopedia, drawn with a fixed seed. |
| R2-5: masking sensitivity, uncertainty | AOPC re-evaluated with three replacement strategies (`<mask>`, deletion, `...`); 10 random rankings per text; Wilcoxon tests and bootstrap 95% CIs per dataset, language, correctness and confidence stratum. |

**Kaggle**: *Add Input* SemEval-2025 Task 11, PRDECT-ID, and the **same embedding-cache dataset as notebook 03**. Accelerator **None (CPU)**, like notebook 03 (≈ 2–3 h, no GPU quota): XGBoost trees depend on the CPU, so the seed-42 models are identical to notebook 03 only on the same kind of machine; a GPU session (different CPU) stops at the consistency check. Internet **On** (downloads multilingual-E5). Outputs: `/kaggle/working/rerun_shap` (+ zip).

In [ ]:
# ============================================================
# 0. INSTALL & IMPORTS
# ============================================================
!pip -q install iterative-stratification shap==0.51.0

import os, sys, re, time, json, glob, random, warnings, gc, platform, shutil, hashlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy import stats
import joblib, shap

import sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
import xgboost as xgb
import transformers
from transformers import AutoTokenizer, AutoModel
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit, MultilabelStratifiedKFold

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NOTEBOOK_T0 = time.time()
print("shap", shap.__version__, "| xgboost", xgb.__version__, "| sklearn", sklearn.__version__, "| device", DEVICE)

## 1. Configuration

In [ ]:
# ============================================================
# 1. CONFIG (pipeline settings identical to notebook 03)
# ============================================================
SMOKE_TEST = bool(int(os.environ.get("MEXBAL_SMOKE", "0")))
SEEDS = [42]; LC_SEEDS = [42]; SEED_PRIMARY = 42; LC_SIZES = [100]
VAL_FRAC, TEST_FRAC = 0.15, 0.20
N_OOF_FOLDS = 5
FOLD_STRATIFY_BY_LANG, W_OBJECTIVE, XGB_NTHREAD, XGB_INPUT_DECIMALS = True, "pooled", 4, 5

ENCODERS = {"e5": dict(name=os.environ.get("MEXBAL_FT_MODEL", "intfloat/multilingual-e5-base"), prefix="query: ")}
MAIN_ENCODER = "e5"
MAX_LEN, EMBED_BATCH = 128, 16

WEIGHT_GRID    = np.arange(0.0, 1.0001, 0.05)
THRESHOLD_GRID = np.arange(0.15, 0.8501, 0.01)
MIN_PRECISION, MAX_PPR_RATIO, TIE_TOL = 0.20, 3.0, 0.005
ALPHA_GRID, ALPHA_PASSES = np.arange(0.50, 3.0001, 0.05), 3
XGB_COMMON = dict(max_depth=3, min_child_weight=5, learning_rate=0.05, n_estimators=200,
                  subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.5, tree_method="hist", verbosity=0)
XGB_RUNTIME = {"n_jobs": XGB_NTHREAD}
XGB_PARAMS_BINARY = dict(objective="binary:logistic", eval_metric="logloss", **XGB_COMMON)
XGB_PARAMS_MULTI  = dict(objective="multi:softprob", eval_metric="mlogloss", **XGB_COMMON)

# ---- SHAP / faithfulness (prespecified) ------------------------
SAMPLE_SEED      = 2026
N_PER_STRATUM    = {"SemEval": 8, "Tokopedia": 15}     # per stratum (and per SemEval language)
MIN_TOKENS       = 2
MASK_TOKEN       = "<mask>"                            # same token in SHAP and in the perturbation tests
TOKEN_PATTERN    = r"\W+"
SHAP_MAX_EVALS, SHAP_BATCH_SIZE = 300, 32
FRACTIONS        = [0.10, 0.25, 0.50]
N_RANDOM         = 10
REPLACEMENTS     = {"mask": MASK_TOKEN, "delete": "", "ellipsis": "..."}   # masking-sensitivity analysis
BOOT_REPS        = 2000
if SMOKE_TEST:
    N_PER_STRATUM, SHAP_MAX_EVALS, N_RANDOM, BOOT_REPS = {"SemEval": 1, "Tokopedia": 2}, 40, 2, 200

INPUT_ROOTS = ("/kaggle/input", os.environ.get("MEXBAL_DATA", "/nonexistent"))
def _find(pattern, roots=INPUT_ROOTS):
    for r in roots:
        hits = sorted(glob.glob(os.path.join(r, "**", pattern), recursive=True))
        if hits: return hits[0]
    return None
def _semeval_root():
    hits = [h for r in INPUT_ROOTS for h in sorted(glob.glob(os.path.join(r, "**", "track_a/train/sun.csv"), recursive=True))]
    hits.sort(key=lambda h: "public_data_test" not in h)
    for h in hits:
        root = Path(h).parents[2]
        need = [root / "track_a/dev/sun.csv", root / "track_c/dev/jav.csv", root / "track_c/dev/ind.csv"]
        if all(f.exists() for f in need) and all({"anger", "joy"} <= set(pd.read_csv(f, nrows=1).columns) for f in need):
            return str(root)
    return None
SEMEVAL_ROOT = _semeval_root(); assert SEMEVAL_ROOT, "SemEval data not found"
TOKOPEDIA_CSV = _find("PRDECT-ID Dataset.csv") or _find("*PRDECT*.csv") or _find("Tokopedia Review.csv"); assert TOKOPEDIA_CSV
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("MEXBAL_WORK", "./work")
OUT_ROOT = os.path.join(WORK, "rerun_shap"); CACHE_OUT = os.path.join(WORK, "embedding_cache")
_cache_hit = _find("tokopedia_e5.npz"); CACHE_IN = str(Path(_cache_hit).parent) if _cache_hit else None
for d in [OUT_ROOT, CACHE_OUT, f"{OUT_ROOT}/figures"]: Path(d).mkdir(parents=True, exist_ok=True)
print("Cache input:", CACHE_IN, "| output:", OUT_ROOT, "| SMOKE_TEST =", SMOKE_TEST)

def set_seed(seed): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
def append_csv(path, rows):
    new = pd.DataFrame(rows); old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    df = pd.concat([old, new], ignore_index=True); df.to_csv(path, index=False); return df
def done_keys(path, cols):
    if not os.path.exists(path): return set()
    return set(map(tuple, pd.read_csv(path)[cols].astype(str).values.tolist()))
safe = lambda s: re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")

## 2. Data, embeddings, engines (copied verbatim from notebook 03)

In [ ]:
# ============================================================
# 2. DATA (identical to the original notebook) + nested LC order
# ============================================================
SEMEVAL_LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
LANGS = ["sun", "jav", "ind"]          # pooling order of the development partitions

def load_semeval():
    sun = pd.concat([pd.read_csv(f"{SEMEVAL_ROOT}/track_a/train/sun.csv"),
                     pd.read_csv(f"{SEMEVAL_ROOT}/track_a/dev/sun.csv")], ignore_index=True)
    jav = pd.read_csv(f"{SEMEVAL_ROOT}/track_c/dev/jav.csv")
    ind = pd.read_csv(f"{SEMEVAL_ROOT}/track_c/dev/ind.csv")
    data = {"sun": sun.reset_index(drop=True), "jav": jav.reset_index(drop=True), "ind": ind.reset_index(drop=True)}
    for lang, df in data.items():
        assert "text" in df.columns and all(c in df.columns for c in SEMEVAL_LABELS), lang
        df["uid"] = [f"{lang}_{i:05d}" for i in range(len(df))]
    return data

def load_tokopedia():
    df = pd.read_csv(TOKOPEDIA_CSV)[["Customer Review", "Emotion"]].dropna().reset_index(drop=True)
    le = LabelEncoder(); df["label"] = le.fit_transform(df["Emotion"])
    df["uid"] = [f"tok_{i:05d}" for i in range(len(df))]
    return df, list(le.classes_)

def _two_stage(splitter, y, seed):
    # identical to the original notebook: 65 / 15 / 20 split, stratified, random_state = seed in both stages
    hold = VAL_FRAC + TEST_FRAC
    tr, h = next(splitter(n_splits=1, test_size=hold, random_state=seed).split(np.zeros(len(y)), y))
    va, te = next(splitter(n_splits=1, test_size=TEST_FRAC / hold, random_state=seed).split(np.zeros(len(h)), y[h]))
    return np.sort(tr), np.sort(h[va]), np.sort(h[te])

LC_ORDER_SALT = 20261023

def lc_order(seed):
    """Nested, approximately stratified ordering of the training partition of one seed.
    The learning-curve subset of size n is the first n entries, so every smaller subset is
    contained in every larger one (nested design). Each class is shuffled and its members are
    spread uniformly over the ranking, so every prefix keeps the class proportions."""
    tr = TOKO_SPLITS[seed][0]; lab = TOKO_DF.label.values[tr]
    rng = np.random.default_rng(LC_ORDER_SALT + seed)
    key = np.empty(len(tr))
    for c in np.unique(lab):
        pos = np.where(lab == c)[0]; m = len(pos)
        key[pos[rng.permutation(m)]] = (np.arange(m) + rng.random(m)) / m
    return np.asarray(tr)[np.argsort(key, kind="stable")]

SEMEVAL_DATA = load_semeval()
TOKO_DF, TOKO_CLASSES = load_tokopedia()
K_TOK = len(TOKO_CLASSES)

ALL_SEEDS = list(dict.fromkeys(SEEDS + LC_SEEDS + [SEED_PRIMARY]))
SEMEVAL_SPLITS = {(l, s): _two_stage(MultilabelStratifiedShuffleSplit, SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int), s)
                  for l in LANGS for s in ALL_SEEDS}
TOKO_SPLITS = {s: _two_stage(StratifiedShuffleSplit, TOKO_DF["label"].values, s) for s in ALL_SEEDS}
for parts in list(SEMEVAL_SPLITS.values()) + list(TOKO_SPLITS.values()):
    tr, va, te = map(set, parts); assert not (tr & va or tr & te or va & te)

# split manifest (same format as the original run, so the two can be compared row by row)
man = []
for (l, s), parts in SEMEVAL_SPLITS.items():
    for sp, idx in zip(["train", "val", "test"], parts):
        man += [{"dataset": "SemEval", "lang": l, "seed": s, "split": sp, "uid": u} for u in SEMEVAL_DATA[l]["uid"].values[idx]]
for s, parts in TOKO_SPLITS.items():
    for sp, idx in zip(["train", "val", "test"], parts):
        man += [{"dataset": "Tokopedia", "lang": "ind", "seed": s, "split": sp, "uid": u} for u in TOKO_DF["uid"].values[idx]]
pd.DataFrame(man).to_csv(f"{OUT_ROOT}/split_manifest_all.csv", index=False)

# nested learning-curve subsets: subset(n) = first n rows of the order of that seed
LC_ORDER = {s: lc_order(s) for s in LC_SEEDS}
rows = []
for s, o in LC_ORDER.items():
    rows += [{"seed": s, "rank": r, "uid": TOKO_DF.uid.values[i], "label": TOKO_CLASSES[TOKO_DF.label.values[i]]} for r, i in enumerate(o)]
pd.DataFrame(rows).to_csv(f"{OUT_ROOT}/lc_subset_order.csv", index=False)
for s in LC_SEEDS[:1]:
    for n in LC_SIZES[:3]:
        print(f"seed {s} n={n} class counts:", np.bincount(TOKO_DF.label.values[LC_ORDER[s][:n]], minlength=K_TOK))
print("SemEval:", {l: len(d) for l, d in SEMEVAL_DATA.items()}, "| Tokopedia:", len(TOKO_DF), TOKO_CLASSES)
print("Tokopedia seed 42 train/val/test:", [len(x) for x in TOKO_SPLITS[42]])

In [ ]:
# ============================================================
# 3. EMBEDDINGS (identical encoding; reused from the cache when attached)
# ============================================================
def masked_mean_pool(h, mask):
    m = mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode(texts, model, tok, prefix=None, batch=EMBED_BATCH):
    texts = [(prefix or "") + str(x) for x in texts]; rows = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i + batch], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        z = masked_mean_pool(model(**enc).last_hidden_state, enc["attention_mask"])
        rows.append(F.normalize(z, p=2, dim=1).cpu().numpy())
    return np.vstack(rows)

def cached_embeddings(cache_name, texts, enc_key):
    for root in [CACHE_IN, CACHE_OUT]:
        if root and os.path.exists(f"{root}/{cache_name}.npz"):
            emb = np.load(f"{root}/{cache_name}.npz")["embedding"]
            if emb.shape[0] == len(texts):
                print("[cache]", cache_name, emb.shape); return emb
    print("[computing on", DEVICE, "]", cache_name, "— this is slow on CPU; attach the embedding cache if possible")
    cfg = ENCODERS[enc_key]
    tok = AutoTokenizer.from_pretrained(cfg["name"]); model = AutoModel.from_pretrained(cfg["name"]).to(DEVICE).eval()
    emb = encode(texts, model, tok, cfg["prefix"])
    np.savez_compressed(f"{CACHE_OUT}/{cache_name}.npz", embedding=emb)
    del model, tok; gc.collect(); return emb

SEMEVAL_EMB = {k: {l: cached_embeddings(f"semeval_{k}_{l}", SEMEVAL_DATA[l]["text"].tolist(), k) for l in LANGS} for k in ENCODERS}
TOKO_EMB = {k: cached_embeddings(f"tokopedia_{k}", TOKO_DF["Customer Review"].tolist(), k) for k in ENCODERS}
EMB_FINGERPRINT = {f"tokopedia_{k}": hashlib.md5(np.round(v, 6).astype(np.float32).tobytes()).hexdigest()[:12] for k, v in TOKO_EMB.items()}
print("Embedding fingerprints:", EMB_FINGERPRINT)

In [ ]:
# ============================================================
# 4. ENGINES
# ============================================================
def fit_preprocessor(X, seed=42):
    sc = StandardScaler().fit(X)
    pc = PCA(n_components=0.95, random_state=seed).fit(sc.transform(X))
    return (lambda A: pc.transform(sc.transform(A))), sc, pc

def xgb_in(Z):
    return np.round(Z, XGB_INPUT_DECIMALS) if XGB_INPUT_DECIMALS is not None else Z

def new_log():
    return {"constant_fits": 0, "threshold_fallbacks": 0}

class ConstantBinary:
    # used when an emotion has no positive (or no negative) example in a training fold
    def __init__(self, p): self.p = float(p)
    def predict_proba(self, X): return np.c_[np.full(len(X), 1 - self.p), np.full(len(X), self.p)]

def combine(px, pl, w):
    if px is None: return pl
    if pl is None: return px
    return w * px + (1 - w) * pl

# ---------------- multi-label (SemEval) ----------------
def train_ml(X, y, cfg, seed, log=None):
    xm = {} if cfg["use_xgb"] else None; lm = {} if cfg["use_lr"] else None
    for j, e in enumerate(SEMEVAL_LABELS):
        yy = y[:, j]; pos = int(yy.sum()); neg = len(yy) - pos
        degenerate = pos == 0 or neg == 0
        if degenerate and log is not None: log["constant_fits"] += 1
        if cfg["use_xgb"]:
            if degenerate: xm[e] = ConstantBinary(yy.mean())
            else:
                ratio = neg / pos if cfg["cost_sensitive"] else 1.0
                xm[e] = xgb.XGBClassifier(scale_pos_weight=ratio, random_state=seed, n_jobs=XGB_RUNTIME["n_jobs"],
                                          **XGB_PARAMS_BINARY).fit(xgb_in(X), yy)
        if cfg["use_lr"]:
            if degenerate: lm[e] = ConstantBinary(yy.mean())
            else:
                lm[e] = LogisticRegression(C=cfg["lr_C"], max_iter=1000, solver="lbfgs",
                                           class_weight="balanced" if cfg["lr_balanced"] else None,
                                           random_state=seed).fit(X, yy)
    return xm, lm

def proba_ml(X, xm, lm):
    px = np.column_stack([xm[e].predict_proba(xgb_in(X))[:, 1] for e in SEMEVAL_LABELS]) if xm is not None else None
    pl = np.column_stack([lm[e].predict_proba(X)[:, 1] for e in SEMEVAL_LABELS]) if lm is not None else None
    return px, pl

def ml_score(y, yp, groups=None, objective="pooled"):
    if objective == "pooled" or groups is None:
        return f1_score(y, yp, average="macro", zero_division=0)
    return float(np.mean([f1_score(y[groups == g], yp[groups == g], average="macro", zero_division=0) for g in np.unique(groups)]))

def tune_w_ml(px, pl, y, groups=None, objective="pooled"):
    # ensemble weight at a fixed 0.5 threshold; ties -> smallest weight (first maximum)
    best = (0.5, -np.inf)
    for w in WEIGHT_GRID:
        s = ml_score(y, (combine(px, pl, w) >= 0.5).astype(int), groups, objective)
        if s > best[1]: best = (float(w), s)
    return best[0]

def tune_th(p, y, log=None):
    # one global threshold per emotion; constraints: precision >= 0.20 and positive rate <= max(3*base, base+0.05);
    # ties within 0.005 -> highest threshold; if no threshold satisfies the constraints -> unconstrained best F1
    out = []
    for j in range(p.shape[1]):
        base = float(y[:, j].mean()); cap = min(1.0, max(MAX_PPR_RATIO * base, base + 0.05)); sc = []
        for t in THRESHOLD_GRID:
            yp = (p[:, j] >= t).astype(int)
            if yp.mean() > cap: sc.append(-np.inf); continue
            prec = y[:, j][yp == 1].mean() if yp.sum() else 0
            if prec < MIN_PRECISION: sc.append(-np.inf); continue
            sc.append(f1_score(y[:, j], yp, zero_division=0))
        a = np.asarray(sc)
        if not np.isfinite(a).any():
            if log is not None: log["threshold_fallbacks"] += 1
            a = np.asarray([f1_score(y[:, j], (p[:, j] >= t).astype(int), zero_division=0) for t in THRESHOLD_GRID])
        out.append(float(THRESHOLD_GRID[np.where(a >= np.nanmax(a) - TIE_TOL)[0][-1]]))
    return np.asarray(out)

def oof_ml(X, y, cfg, seed, strat=None, log=None):
    px = np.zeros(y.shape) if cfg["use_xgb"] else None; pl = np.zeros(y.shape) if cfg["use_lr"] else None
    S = y if strat is None else np.c_[y, strat]
    for a, b in MultilabelStratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), S):
        tf, _, _ = fit_preprocessor(X[a], seed); xm, lm = train_ml(tf(X[a]), y[a], cfg, seed, log)
        qx, ql = proba_ml(tf(X[b]), xm, lm)
        if px is not None: px[b] = qx
        if pl is not None: pl[b] = ql
    return px, pl

# ---------------- single-label (Tokopedia) ----------------
def train_sl(X, y, cfg, seed):
    xm = lm = None
    if cfg["use_xgb"]:
        sw = None
        if cfg["cost_sensitive"]:
            cnt = np.bincount(y, minlength=K_TOK); wt = len(y) / (K_TOK * np.maximum(cnt, 1)); sw = wt[y]
        xm = xgb.XGBClassifier(num_class=K_TOK, random_state=seed, n_jobs=XGB_RUNTIME["n_jobs"],
                               **XGB_PARAMS_MULTI).fit(xgb_in(X), y, sample_weight=sw)
    if cfg["use_lr"]:
        lm = LogisticRegression(C=cfg["lr_C"], max_iter=1000, solver="lbfgs",
                                class_weight="balanced" if cfg["lr_balanced"] else None, random_state=seed).fit(X, y)
    return xm, lm

def proba_sl(X, xm, lm):
    return (xm.predict_proba(xgb_in(X)) if xm is not None else None), (lm.predict_proba(X) if lm is not None else None)

def oof_sl(X, y, cfg, seed):
    px = np.zeros((len(y), K_TOK)) if cfg["use_xgb"] else None; pl = np.zeros((len(y), K_TOK)) if cfg["use_lr"] else None
    for a, b in StratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y):
        tf, _, _ = fit_preprocessor(X[a], seed); xm, lm = train_sl(tf(X[a]), y[a], cfg, seed)
        qx, ql = proba_sl(tf(X[b]), xm, lm)
        if px is not None: px[b] = qx
        if pl is not None: pl[b] = ql
    return px, pl

def tune_w_sl(px, pl, y):
    best = (0.5, -np.inf)
    for w in WEIGHT_GRID:
        s = f1_score(y, np.argmax(combine(px, pl, w), 1), average="macro", zero_division=0)
        if s > best[1]: best = (float(w), s)
    return best[0]

def tune_alpha(p, y):
    # class-prior coefficients by coordinate ascent; ties within 0.005 -> value closest to 1
    alpha = np.ones(K_TOK)
    for _ in range(ALPHA_PASSES):
        for c in range(K_TOK):
            sc = []
            for a in ALPHA_GRID:
                q = alpha.copy(); q[c] = a
                sc.append(f1_score(y, np.argmax(p * q, 1), average="macro", zero_division=0))
            sc = np.asarray(sc); cand = np.where(sc >= sc.max() - TIE_TOL)[0]
            alpha[c] = ALPHA_GRID[cand[np.argmin(np.abs(ALPHA_GRID[cand] - 1))]]
    return alpha

In [ ]:
# ============================================================
# 5. CONFIGURATIONS
# ============================================================
def C(name, encoder, use_xgb=True, use_lr=True, cost_sensitive=False, lr_balanced=False, lr_C=1.0):
    return dict(name=name, encoder=encoder, use_xgb=use_xgb, use_lr=use_lr,
                cost_sensitive=cost_sensitive, lr_balanced=lr_balanced, lr_C=lr_C)

W = dict(cost_sensitive=True, lr_balanced=True)            # class weighting (both branches)
REF = "E5 Hybrid (imbalance-aware)"                        # = ME-XBAL in the old result files
CONFIGS = [
    # --- the 11 models of the manuscript (old labels: ME-XBAL (XLM-R) -> XLM-R Hybrid (imbalance-aware), ME-XBAL -> REF)
    C("XLM-R LR", "xlmr", use_xgb=False), C("XLM-R XGB", "xlmr", use_lr=False), C("XLM-R Hybrid", "xlmr"),
    C("IndoBERT LR", "indobert", use_xgb=False), C("IndoBERT XGB", "indobert", use_lr=False), C("IndoBERT Hybrid", "indobert"),
    C("E5 LR", "e5", use_xgb=False), C("E5 XGB", "e5", use_lr=False), C("E5 Hybrid", "e5"),
    C("XLM-R Hybrid (imbalance-aware)", "xlmr", **W, lr_C=0.1), C(REF, "e5", **W, lr_C=0.1),
    # --- new controls (R2-3, R1-3): 2 x 2 factorials (weighting x C) for the LR-only and the hybrid head
    C("E5 LR (C=0.1)", "e5", use_xgb=False, lr_C=0.1),
    C("E5 LR (weighted)", "e5", use_xgb=False, lr_balanced=True),
    C("E5 LR (imbalance-aware)", "e5", use_xgb=False, lr_balanced=True, lr_C=0.1),
    C("E5 XGB (cost-sensitive)", "e5", use_lr=False, cost_sensitive=True),
    C("E5 Hybrid (C=0.1)", "e5", lr_C=0.1),
    C("E5 Hybrid (weighted)", "e5", **W),
    C("XLM-R LR (imbalance-aware)", "xlmr", use_xgb=False, lr_balanced=True, lr_C=0.1),
]
CFG = {c["name"]: c for c in CONFIGS}
assert len(CFG) == len(CONFIGS)
print(len(CONFIGS), "configurations")

# ============================================================
# 6. RUNNERS
# ============================================================
def lang_onehot(groups):
    return np.eye(len(LANGS), dtype=int)[groups]

def run_sem(cfg, seed, fold_by_lang=None, w_objective=None):
    fold_by_lang = FOLD_STRATIFY_BY_LANG if fold_by_lang is None else fold_by_lang
    w_objective = w_objective or W_OBJECTIVE
    E = SEMEVAL_EMB[cfg["encoder"]]; log = new_log()
    dev = {l: np.r_[SEMEVAL_SPLITS[(l, seed)][0], SEMEVAL_SPLITS[(l, seed)][1]] for l in LANGS}
    X = np.vstack([E[l][dev[l]] for l in LANGS])
    y = np.vstack([SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[dev[l]] for l in LANGS])
    g = np.concatenate([np.full(len(dev[l]), i) for i, l in enumerate(LANGS)])
    px, pl = oof_ml(X, y, cfg, seed, lang_onehot(g) if fold_by_lang else None, log)
    w = tune_w_ml(px, pl, y, g, w_objective) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    th = tune_th(combine(px, pl, w), y, log)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_ml(tf(X), y, cfg, seed, log)
    rows, art, YT, YP = [], {"weight_xgb": w, "thresholds": th, "log": log, "n_dev": {l: len(dev[l]) for l in LANGS}}, [], []
    for l in LANGS:
        te = SEMEVAL_SPLITS[(l, seed)][2]
        yt = SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[te]
        pt = combine(*proba_ml(tf(E[l][te]), xm, lm), w); yp = (pt >= th[None, :]).astype(int)
        rows.append({"model": cfg["name"], "seed": seed, "lang": l, "n_test": len(te), "weight_xgb": w,
                     "macro_f1": f1_score(yt, yp, average="macro", zero_division=0)})
        art[l] = {"test_uid": SEMEVAL_DATA[l]["uid"].values[te], "y_true": yt, "y_pred": yp, "proba": pt.astype(np.float32)}
        YT.append(yt); YP.append(yp)
    YT, YP = np.vstack(YT), np.vstack(YP)
    rows.append({"model": cfg["name"], "seed": seed, "lang": "pooled", "n_test": len(YT), "weight_xgb": w,
                 "macro_f1": f1_score(YT, YP, average="macro", zero_division=0)})
    rows.append({"model": cfg["name"], "seed": seed, "lang": "average", "n_test": len(YT), "weight_xgb": w,
                 "macro_f1": float(np.mean([r["macro_f1"] for r in rows[:3]]))})
    for r in rows: r.update(constant_fits=log["constant_fits"], threshold_fallbacks=log["threshold_fallbacks"])
    return rows, art

def _tok_eval(cfg, seed, xm, lm, tf, w, alpha, te):
    E = TOKO_EMB[cfg["encoder"]]; yt = TOKO_DF.label.values[te]
    pt = combine(*proba_sl(tf(E[te]), xm, lm), w); yp = np.argmax(pt * alpha, 1)
    row = {"model": cfg["name"], "seed": seed, "n_test": len(te), "weight_xgb": w,
           "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
           "micro_f1": f1_score(yt, yp, average="micro", zero_division=0), "accuracy": accuracy_score(yt, yp),
           "alpha": json.dumps([round(float(a), 2) for a in alpha])}
    art = {"weight_xgb": w, "alpha": alpha, "test_uid": TOKO_DF.uid.values[te], "y_true": yt, "y_pred": yp,
           "proba": pt.astype(np.float32)}
    return row, art

def run_tok(cfg, seed, fit_idx=None):
    """Fit AND select (out-of-fold) on fit_idx (default: train + validation = development partition)."""
    E = TOKO_EMB[cfg["encoder"]]; tr, va, te = TOKO_SPLITS[seed]
    dev = np.r_[tr, va] if fit_idx is None else np.asarray(fit_idx)
    X = E[dev]; y = TOKO_DF.label.values[dev]
    px, pl = oof_sl(X, y, cfg, seed)
    w = tune_w_sl(px, pl, y) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    alpha = tune_alpha(combine(px, pl, w), y)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_sl(tf(X), y, cfg, seed)
    return _tok_eval(cfg, seed, xm, lm, tf, w, alpha, te)

def run_tok_valsel(cfg, seed, fit_idx, sel_idx):
    """Fit on fit_idx only; select the ensemble weight and class priors on sel_idx (regime B)."""
    E = TOKO_EMB[cfg["encoder"]]; te = TOKO_SPLITS[seed][2]
    X = E[fit_idx]; y = TOKO_DF.label.values[fit_idx]; ys = TOKO_DF.label.values[sel_idx]
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_sl(tf(X), y, cfg, seed)
    px, pl = proba_sl(tf(E[sel_idx]), xm, lm)
    w = tune_w_sl(px, pl, ys) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    alpha = tune_alpha(combine(px, pl, w), ys)
    return _tok_eval(cfg, seed, xm, lm, tf, w, alpha, te)

## 3. Seed-42 models of the imbalance-aware E5 hybrid and raw-text predictors

In [ ]:
# ============================================================
# 3. BUNDLES (exactly the steps of run_sem / run_tok in notebook 03)
# ============================================================
cfg = CFG[REF]; seed = SEED_PRIMARY

def fit_bundle_sem():
    E = SEMEVAL_EMB["e5"]; log = new_log()
    dev = {l: np.r_[SEMEVAL_SPLITS[(l, seed)][0], SEMEVAL_SPLITS[(l, seed)][1]] for l in LANGS}
    X = np.vstack([E[l][dev[l]] for l in LANGS]); y = np.vstack([SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[dev[l]] for l in LANGS])
    g = np.concatenate([np.full(len(dev[l]), i) for i, l in enumerate(LANGS)])
    px, pl = oof_ml(X, y, cfg, seed, lang_onehot(g) if FOLD_STRATIFY_BY_LANG else None, log)
    w = tune_w_ml(px, pl, y, g, W_OBJECTIVE); th = tune_th(combine(px, pl, w), y, log)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_ml(tf(X), y, cfg, seed, log)
    return dict(tf=tf, xm=xm, lm=lm, w=w, th=th)

def fit_bundle_tok():
    tr, va, te = TOKO_SPLITS[seed]; dev = np.r_[tr, va]; X = TOKO_EMB["e5"][dev]; y = TOKO_DF.label.values[dev]
    px, pl = oof_sl(X, y, cfg, seed); w = tune_w_sl(px, pl, y); alpha = tune_alpha(combine(px, pl, w), y)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_sl(tf(X), y, cfg, seed)
    return dict(tf=tf, xm=xm, lm=lm, w=w, alpha=alpha)

SEM_B, TOK_B = fit_bundle_sem(), fit_bundle_tok()

# consistency with notebook 03 (seed 42 of E5 Hybrid (imbalance-aware)); identical code => identical numbers
REF_NB03 = {"sun": 0.449803, "jav": 0.496471, "ind": 0.403881, "tok": 0.634605}
check = {}
for l in LANGS:
    te = SEMEVAL_SPLITS[(l, seed)][2]; yt = SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[te]
    P = combine(*proba_ml(SEM_B["tf"](SEMEVAL_EMB["e5"][l][te]), SEM_B["xm"], SEM_B["lm"]), SEM_B["w"])
    check[l] = f1_score(yt, (P >= SEM_B["th"][None, :]).astype(int), average="macro", zero_division=0)
te = TOKO_SPLITS[seed][2]; P = combine(*proba_sl(TOK_B["tf"](TOKO_EMB["e5"][te]), TOK_B["xm"], TOK_B["lm"]), TOK_B["w"])
check["tok"] = f1_score(TOKO_DF.label.values[te], np.argmax(P * TOK_B["alpha"], 1), average="macro", zero_division=0)
print({k: round(v, 6) for k, v in check.items()}, "| notebook 03:", REF_NB03)
if not SMOKE_TEST:
    CPU_NB03 = "Intel(R) Xeon(R) CPU @ 2.20GHz, 4 CPUs (Kaggle, accelerator None)"
    _cpu = subprocess.run(["bash", "-c", "lscpu | grep -E 'Model name|^CPU\\(s\\)'"], capture_output=True, text=True).stdout.strip()
    print("This machine:", _cpu.replace("\n", " | "), "| notebook 03 ran on:", CPU_NB03)
    assert all(abs(check[k] - REF_NB03[k]) < 1e-5 for k in REF_NB03), (
        "Seed-42 models differ from notebook 03. With the same embedding cache this is the XGBoost hardware dependence "
        "(different CPU). Run this notebook with Accelerator = None (CPU), like notebook 03.")
print("weight_xgb SemEval", SEM_B["w"], "| thresholds", SEM_B["th"].round(2), "| Tokopedia weight", TOK_B["w"], "alpha", TOK_B["alpha"])

ENC_TOK = AutoTokenizer.from_pretrained(ENCODERS["e5"]["name"])
ENC_MODEL = AutoModel.from_pretrained(ENCODERS["e5"]["name"]).to(DEVICE).eval()
PREFIX = ENCODERS["e5"]["prefix"]

def embed_raw(texts):
    # the prefix is added here, i.e. AFTER masking: it is never masked
    return encode(list(texts), ENC_MODEL, ENC_TOK, PREFIX, batch=SHAP_BATCH_SIZE)

def P_sem(texts): Z = SEM_B["tf"](embed_raw(texts)); return combine(*proba_ml(Z, SEM_B["xm"], SEM_B["lm"]), SEM_B["w"])
def P_tok(texts): Z = TOK_B["tf"](embed_raw(texts)); return combine(*proba_sl(Z, TOK_B["xm"], TOK_B["lm"]), TOK_B["w"])

def adjusted(P):            # Tokopedia decision-adjusted scores q = alpha*P / sum
    Q = P * TOK_B["alpha"][None, :]; return Q / Q.sum(1, keepdims=True)

# raw-text vs cached-embedding agreement (recorded; the raw path is the one that is explained)
_te = TOKO_SPLITS[seed][2][:16]
_pc = combine(*proba_sl(TOK_B["tf"](TOKO_EMB["e5"][_te]), TOK_B["xm"], TOK_B["lm"]), TOK_B["w"])
_pr = P_tok(TOKO_DF["Customer Review"].values[_te].tolist())
RAW_VS_CACHE = float(np.abs(_pc - _pr).max()); print("max |P(raw text) - P(cached embedding)| =", RAW_VS_CACHE)

## 4. Prespecified stratified sample (correct / error × confidence tercile)

In [ ]:
# ============================================================
# 4. SAMPLE
# ============================================================
def regex_spans(text):
    spans, pos = [], 0
    for m in re.finditer(TOKEN_PATTERN, text):
        if pos < m.start(): spans.append((pos, m.start()))
        pos = m.end()
    if pos < len(text): spans.append((pos, len(text)))
    return spans

def tercile(v):
    q1, q2 = np.quantile(v, [1 / 3, 2 / 3]); return np.where(v <= q1, "low", np.where(v <= q2, "mid", "high"))

cand = []
for l in LANGS:   # SemEval: target c = argmax(P_c - tau_c) computed from the raw-text path
    te = SEMEVAL_SPLITS[(l, seed)][2]; texts = SEMEVAL_DATA[l]["text"].astype(str).values[te]
    yt = SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[te]; P = P_sem(texts)
    c = np.argmax(P - SEM_B["th"][None, :], 1); conf = P[np.arange(len(te)), c]
    pos = P[np.arange(len(te)), c] >= SEM_B["th"][c]; gold = yt[np.arange(len(te)), c] == 1
    status = np.where(pos & gold, "TP", np.where(pos & ~gold, "FP", np.where(~pos & gold, "FN", "TN")))
    exact = (yt == (P >= SEM_B["th"][None, :]).astype(int)).all(1)
    for k, i in enumerate(te):
        cand.append(dict(dataset="SemEval", lang=l, id=SEMEVAL_DATA[l]["uid"].values[i], text=texts[k], target_idx=int(c[k]),
                         target=SEMEVAL_LABELS[c[k]], gold="|".join(e for e, v in zip(SEMEVAL_LABELS, yt[k]) if v) or "none",
                         correct=bool(gold[k]), decision_status=status[k], label_set_exact=bool(exact[k]),
                         score=float(conf[k]), threshold=float(SEM_B["th"][c[k]])))
    for r, t in zip(cand[-len(te):], tercile(conf)): r["confidence"] = t
te = TOKO_SPLITS[seed][2]; texts = TOKO_DF["Customer Review"].astype(str).values[te]; yt = TOKO_DF.label.values[te]
P = P_tok(texts); Q = adjusted(P); k_ = np.argmax(Q, 1); conf = Q[np.arange(len(te)), k_]
for k, i in enumerate(te):
    cand.append(dict(dataset="Tokopedia", lang="ind", id=TOKO_DF.uid.values[i], text=texts[k], target_idx=int(k_[k]),
                     target=TOKO_CLASSES[k_[k]], gold=TOKO_CLASSES[yt[k]], correct=bool(k_[k] == yt[k]),
                     decision_status="correct" if k_[k] == yt[k] else "error", label_set_exact=bool(k_[k] == yt[k]),
                     score=float(conf[k]), threshold=np.nan))
for r, t in zip(cand[-len(te):], tercile(conf)): r["confidence"] = t
cand = pd.DataFrame(cand); cand["n_tokens"] = cand.text.map(lambda t: len(regex_spans(t)))

rng = np.random.default_rng(SAMPLE_SEED); picks = []
for (ds, l, corr, cf), g in cand[cand.n_tokens >= MIN_TOKENS].groupby(["dataset", "lang", "correct", "confidence"], sort=True):
    n = min(len(g), N_PER_STRATUM[ds]); picks.append(g.iloc[np.sort(rng.choice(len(g), n, replace=False))])
SAMPLE = pd.concat(picks).reset_index(drop=True)
SAMPLE["stratum_available"] = SAMPLE.apply(lambda r: int(((cand.dataset == r.dataset) & (cand.lang == r.lang) & (cand.correct == r.correct)
                                                          & (cand.confidence == r.confidence) & (cand.n_tokens >= MIN_TOKENS)).sum()), axis=1)
cand.drop(columns=["text"]).to_csv(f"{OUT_ROOT}/candidates_seed42.csv", index=False)
SAMPLE.to_csv(f"{OUT_ROOT}/sample_manifest.csv", index=False)
print(SAMPLE.groupby(["dataset", "lang", "correct", "confidence"]).size().unstack().fillna(0).astype(int))

## 5. SHAP (Partition explainer, `<mask>` masker) and faithfulness tests

In [ ]:
# ============================================================
# 5. SHAP + AOPC + decision flips (resumable per text)
# ============================================================
MASKER = shap.maskers.Text(TOKEN_PATTERN, mask_token=MASK_TOKEN, collapse_mask_token=False)
SEEN = []
def score_fn(ds, t):
    def f(texts):
        texts = [texts] if isinstance(texts, str) else [str(s) for s in np.asarray(texts).reshape(-1)]
        if len(SEEN) < 50: SEEN.extend(texts[:5])
        return P_sem(texts)[:, t] if ds == "SemEval" else adjusted(P_tok(texts))[:, t]
    return f

def replace(text, spans, keep, token):
    keep = set(int(i) for i in keep); parts, pos = [], 0
    for i, (s, e) in enumerate(spans):
        parts.append(text[pos:s]); parts.append(text[s:e] if i in keep else token); pos = e
    parts.append(text[pos:]); return "".join(parts)

def text_seed(uid): return int(hashlib.md5(uid.encode()).hexdigest()[:8], 16)

PER_TEXT, SHAP_JSONL, EXCL = f"{OUT_ROOT}/faithfulness_per_text.csv", f"{OUT_ROOT}/shap_values.jsonl", f"{OUT_ROOT}/exclusions.csv"
done = done_keys(PER_TEXT, ["id"]) | done_keys(EXCL, ["id"])
t0 = time.time()
for r in tqdm(SAMPLE.itertuples(), total=len(SAMPLE), desc="SHAP"):
    if (r.id,) in done: continue
    spans = regex_spans(r.text); n = len(spans)
    exp = shap.Explainer(score_fn(r.dataset, r.target_idx), MASKER, algorithm="partition")([r.text], max_evals=max(SHAP_MAX_EVALS, 2 * n + 1))[0]
    phi = np.asarray(exp.values).reshape(-1)
    if len(phi) != n:
        append_csv(EXCL, [{"id": r.id, "dataset": r.dataset, "lang": r.lang, "reason": f"token mismatch: SHAP {len(phi)} vs spans {n}"}]); continue
    with open(SHAP_JSONL, "a") as fh:
        fh.write(json.dumps({"id": r.id, "dataset": r.dataset, "lang": r.lang, "target": r.target, "tokens": [str(t) for t in exp.data],
                             "values": [float(v) for v in phi], "base_value": float(np.asarray(exp.base_values).reshape(-1)[0])}) + "\n")
    rs = np.random.default_rng(text_seed(r.id))
    rankings = {"abs": np.argsort(-np.abs(phi), kind="stable"), "signed": np.argsort(-phi, kind="stable")}
    rand = [rs.permutation(n) for _ in range(N_RANDOM)]
    ks = [min(n, max(1, int(round(f * n)))) for f in FRACTIONS]
    # build every perturbed text once, score them in one batch
    batch, index = [r.text], {}
    for rep, tok in REPLACEMENTS.items():
        index[(rep, "empty")] = len(batch); batch.append(replace(r.text, spans, [], tok))
        for name, order in list(rankings.items()) + [(f"rand{j}", o) for j, o in enumerate(rand)]:
            for fi, k in enumerate(ks):
                top = set(order[:k].tolist())
                index[(rep, name, fi, "del")] = len(batch); batch.append(replace(r.text, spans, [i for i in range(n) if i not in top], tok))
                index[(rep, name, fi, "ins")] = len(batch); batch.append(replace(r.text, spans, sorted(top), tok))
    if r.dataset == "SemEval":
        Pb = P_sem(batch); S = {"score": Pb[:, r.target_idx]}; DEC = Pb[:, r.target_idx] >= SEM_B["th"][r.target_idx]
    else:
        Pb = P_tok(batch); Qb = adjusted(Pb); S = {"score": Qb[:, r.target_idx], "raw": Pb[:, r.target_idx]}
        DEC = np.argmax(Qb, 1) == r.target_idx
    row = {"id": r.id, "dataset": r.dataset, "lang": r.lang, "target": r.target, "correct": r.correct, "confidence": r.confidence,
           "decision_status": r.decision_status, "n_tokens": n, "score_full": float(S["score"][0]), "decision_full": bool(DEC[0]),
           "random_seed": text_seed(r.id)}
    for rep in REPLACEMENTS:
        e = index[(rep, "empty")]
        for fn, s in S.items():
            for name in ["abs", "signed"] + [f"rand{j}" for j in range(N_RANDOM)]:
                d = np.mean([s[0] - s[index[(rep, name, fi, "del")]] for fi in range(len(ks))])
                ins = np.mean([s[index[(rep, name, fi, "ins")]] - s[e] for fi in range(len(ks))])
                key = name if not name.startswith("rand") else "rand"
                row.setdefault(f"{rep}|{fn}|{key}|del", []).append(d); row.setdefault(f"{rep}|{fn}|{key}|ins", []).append(ins)
        for name in ["abs", "signed"] + [f"rand{j}" for j in range(N_RANDOM)]:
            fl_d = np.mean([DEC[0] != DEC[index[(rep, name, fi, "del")]] for fi in range(len(ks))])
            fl_i = np.mean([DEC[index[(rep, name, fi, "ins")]] == DEC[0] for fi in range(len(ks))])
            key = name if not name.startswith("rand") else "rand"
            row.setdefault(f"{rep}|flip|{key}|del", []).append(fl_d); row.setdefault(f"{rep}|flip|{key}|ins", []).append(fl_i)
    row = {k: (float(np.mean(v)) if isinstance(v, list) else v) for k, v in row.items()}
    append_csv(PER_TEXT, [row])
print(f"SHAP finished in {(time.time() - t0) / 60:.1f} min")
print("examples of strings sent to the model during SHAP:", SEEN[:4])
assert any(MASK_TOKEN in s for s in SEEN), "SHAP masker did not use the mask token"

## 6. Summary: SHAP vs. random, by variant and stratum

In [ ]:
# ============================================================
# 6. SUMMARY
# ============================================================
pt = pd.read_csv(PER_TEXT)
def boot_ci(d, reps=BOOT_REPS, seed=SAMPLE_SEED):
    d = np.asarray(d, float); rng = np.random.default_rng(seed)
    m = d[rng.integers(0, len(d), (reps, len(d)))].mean(1); return np.quantile(m, [.025, .975])
rows = []
groups = [("all", pt)] + [(f"lang={l}", g) for l, g in pt[pt.dataset == "SemEval"].groupby("lang")] + \
         [(f"correct={c}", g) for c, g in pt.groupby("correct")] + [(f"confidence={c}", g) for c, g in pt.groupby("confidence")]
for ds, D in pt.groupby("dataset"):
    for gname, G in [("all", D)] + [(f"lang={l}", g) for l, g in D.groupby("lang") if ds == "SemEval"] + \
                    [(f"correct={c}", g) for c, g in D.groupby("correct")] + [(f"confidence={c}", g) for c, g in D.groupby("confidence")]:
        for col in [c for c in D.columns if c.count("|") == 3 and "|rand|" not in c]:
            rep, fn, rank, test = col.split("|"); rc = f"{rep}|{fn}|rand|{test}"
            a, b = G[col].values, G[rc].values; d = a - b
            if len(d) < 2: continue
            lo, hi = boot_ci(d)
            p = float(stats.wilcoxon(a, b).pvalue) if np.any(d != 0) else 1.0
            rows.append(dict(dataset=ds, group=gname, n=len(d), replacement=rep, function=fn, ranking=rank, test=test,
                             shap_mean=a.mean(), random_mean=b.mean(), delta=d.mean(), ci95_low=lo, ci95_high=hi,
                             shap_better=int((d > 0).sum()), wilcoxon_p=p))
summ = pd.DataFrame(rows); summ.to_csv(f"{OUT_ROOT}/faithfulness_summary.csv", index=False)
pd.set_option("display.width", 250)
main = summ[(summ.group == "all") & (summ.replacement == "mask") & (summ.ranking == "abs")]
print(main[["dataset", "function", "test", "n", "shap_mean", "random_mean", "delta", "ci95_low", "ci95_high", "wilcoxon_p"]].round(4).to_string(index=False))
print(summ[(summ.replacement == "mask") & (summ.ranking == "abs") & (summ.function != "flip") & (summ.test == "del") & (summ.group != "all")]
      [["dataset", "group", "function", "n", "delta", "ci95_low", "ci95_high", "wilcoxon_p"]].round(4).to_string(index=False))
if os.path.exists(EXCL): print("Excluded:", pd.read_csv(EXCL).to_string(index=False))

## 7. Example figures (most confident correct text per SemEval language and on Tokopedia)

In [ ]:
# ============================================================
# 7. FIGURES (same style as Figs. 3-6; blue = supports the target, red = against)
# ============================================================
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"], "font.size": 10.5})
vals = {}
if os.path.exists(SHAP_JSONL):
    for line in open(SHAP_JSONL): d = json.loads(line); vals[d["id"]] = d
ex_rows = []
for (ds, l), g in pt[pt.correct & (pt.confidence == "high")].groupby(["dataset", "lang"]):
    r = g.sort_values("score_full", ascending=False).iloc[0]; d = vals[r.id]
    df = pd.DataFrame({"token": [t.strip() for t in d["tokens"]], "shap_value": d["values"]})
    df["abs_shap"] = df.shap_value.abs(); df = df.sort_values("abs_shap", ascending=False)
    top = df.head(12).iloc[::-1]
    fig, ax = plt.subplots(figsize=(4.96, 3.2))
    ax.barh(top.token, top.shap_value, color=["#1f4e79" if v > 0 else "#b22222" for v in top.shap_value])
    ax.axvline(0, lw=0.8, color="black"); ax.set_xlabel("SHAP value (contribution to the explained score)"); ax.set_ylabel("Token")
    fig.tight_layout(); name = f"SHAP_{ds}_{l}"; fig.savefig(f"{OUT_ROOT}/figures/{name}.png", dpi=300); plt.close(fig)
    df.to_csv(f"{OUT_ROOT}/figures/{name}.csv", index=False)
    txt = SAMPLE.set_index("id").loc[r.id, "text"]
    ex_rows.append(dict(figure=name, id=r.id, dataset=ds, lang=l, target=r.target, score=r.score_full, text=txt))
pd.DataFrame(ex_rows).to_csv(f"{OUT_ROOT}/figures/examples_manifest.csv", index=False)
print(pd.DataFrame(ex_rows)[["figure", "id", "target", "score"]].to_string(index=False))

## 8. Environment and archive

In [ ]:
env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__, "shap": shap.__version__,
       "sklearn": sklearn.__version__, "xgboost": xgb.__version__, "numpy": np.__version__, "device": str(DEVICE),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "embedding_cache_input": CACHE_IN,
       "check_vs_nb03": check, "raw_vs_cache_max_abs": RAW_VS_CACHE, "sample_seed": SAMPLE_SEED, "n_per_stratum": N_PER_STRATUM,
       "mask_token": MASK_TOKEN, "token_pattern": TOKEN_PATTERN, "shap_max_evals": SHAP_MAX_EVALS, "fractions": FRACTIONS,
       "n_random": N_RANDOM, "replacements": REPLACEMENTS, "smoke_test": SMOKE_TEST, "runtime_min": round((time.time() - NOTEBOOK_T0) / 60, 1)}
json.dump(env, open(f"{OUT_ROOT}/environment_shap.json", "w"), indent=2, default=str)
shutil.make_archive(f"{WORK}/rerun_shap_outputs", "zip", OUT_ROOT)
print(json.dumps(env, indent=2, default=str))